# Détection des feux de forêt par CNN — ConvNeXt V2 Tiny

**Étudiantes :** sellami mohamed amine - abdesslem meriem - ferhat abderahman kedjour abderahman  
**Encadrant :** dr.sebai meriem  
**Date :** Mai 2026

Ce notebook présente une approche de classification binaire des images de forêt en deux classes : **fire** et **nofire**.  
Le modèle utilisé est **ConvNeXt V2 Tiny pré-entraîné**, adapté ici comme CNN moderne pour la détection d’incendies.


## Importation des bibliothèques


In [ ]:
# Installation de la bibliothèque timm dans Google Colab.
!pip install -q timm


In [ ]:
import os
import time
import random
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from IPython.display import display

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Préparation des données


In [ ]:
DATASET_DIR_DRIVE = "/content/drive/MyDrive/DeepFire/Forest Fire Dataset"
COPY_TO_LOCAL = True
LOCAL_DATASET_DIR = "/content/Forest Fire Dataset"

PROJECT_DIR_DRIVE = "/content/drive/MyDrive/DeepFire/ConvNeXtV2_Tiny_Forest_Fire"
MODELS_DIR = os.path.join(PROJECT_DIR_DRIVE, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR_DRIVE, "results")
FIGURES_DIR = os.path.join(PROJECT_DIR_DRIVE, "figures")
PREDICTIONS_DIR = os.path.join(PROJECT_DIR_DRIVE, "predictions")

for directory in [PROJECT_DIR_DRIVE, MODELS_DIR, RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    os.makedirs(directory, exist_ok=True)

SEED = 42
VAL_RATIO = 0.20
BATCH_SIZE = 16
NUM_WORKERS = 2
IMAGE_SIZE = 224
USE_AMP = True

HEAD_EPOCHS = 3
FINE_EPOCHS = 7
LR_HEAD = 5e-4
LR_FINE = 1e-5
WEIGHT_DECAY = 1e-5
DROPOUT = 0.20
LABEL_SMOOTHING = 0.00
UNFREEZE_STAGES = 2
PATIENCE = 4
DEFAULT_THRESHOLD = 0.50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Dossier du projet:", PROJECT_DIR_DRIVE)


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)


In [ ]:
def copy_dataset_to_local(src, dst):
    src = Path(src)
    dst = Path(dst)

    if not src.exists():
        raise FileNotFoundError(f"Dataset introuvable : {src}")

    if dst.exists():
        return str(dst)

    ignore_patterns = shutil.ignore_patterns(
        "*results*",
        "ConvNeXtV2_Tiny_Forest_Fire",
        "__MACOSX",
        ".ipynb_checkpoints",
    )
    shutil.copytree(src, dst, ignore=ignore_patterns)
    return str(dst)

DATASET_DIR = copy_dataset_to_local(DATASET_DIR_DRIVE, LOCAL_DATASET_DIR) if COPY_TO_LOCAL else DATASET_DIR_DRIVE
TRAINING_DIR = os.path.join(DATASET_DIR, "Training")
TESTING_DIR = os.path.join(DATASET_DIR, "Testing")

print("DATASET_DIR:", DATASET_DIR)
print("Training existe:", os.path.exists(TRAINING_DIR))
print("Testing existe:", os.path.exists(TESTING_DIR))


In [ ]:
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def is_image_file(path):
    return Path(path).suffix.lower() in IMG_EXTENSIONS


def label_from_parent(path):
    parent = Path(path).parent.name.lower().replace(" ", "").replace("-", "").replace("_", "")

    # Tester nofire avant fire, car "nofire" contient "fire".
    if parent in ["nofire", "nonfire", "no"]:
        return 0
    if parent in ["fire", "fires", "flame", "flames"]:
        return 1
    if "nofire" in parent or "nonfire" in parent:
        return 0
    if "fire" in parent:
        return 1
    return None


def collect_labeled_images(training_dir):
    paths, labels = [], []

    for root, _, files in os.walk(training_dir):
        for file_name in files:
            image_path = os.path.join(root, file_name)
            if is_image_file(image_path):
                label = label_from_parent(image_path)
                if label is not None:
                    paths.append(image_path)
                    labels.append(label)

    return np.array(paths), np.array(labels)


all_paths, all_labels = collect_labeled_images(TRAINING_DIR)

if len(all_paths) == 0:
    raise ValueError("Aucune image trouvée. Vérifiez les dossiers Training/fire et Training/nofire.")

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths,
    all_labels,
    test_size=VAL_RATIO,
    stratify=all_labels,
    random_state=SEED,
)

print("Total images:", len(all_paths))
print("Train:", len(train_paths), "| Validation:", len(val_paths))
print("fire:", int(np.sum(all_labels == 1)), "| nofire:", int(np.sum(all_labels == 0)))

split_df = pd.DataFrame({
    "path": np.concatenate([train_paths, val_paths]),
    "label": np.concatenate([train_labels, val_labels]),
    "label_name": ["fire" if y == 1 else "nofire" for y in np.concatenate([train_labels, val_labels])],
    "split": ["train"] * len(train_paths) + ["val"] * len(val_paths),
})

split_path = os.path.join(RESULTS_DIR, "train_val_split.csv")
split_df.to_csv(split_path, index=False)


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class ForestFireDataset(Dataset):
    def __init__(self, paths, labels=None, transform=None):
        self.paths = list(paths)
        self.labels = None if labels is None else list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image_path = self.paths[idx]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.labels is None:
            return image, image_path

        label = torch.tensor(float(self.labels[idx]), dtype=torch.float32)
        return image, label


train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=8),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = ForestFireDataset(train_paths, train_labels, transform=train_transform)
val_ds = ForestFireDataset(val_paths, val_labels, transform=val_transform)

class_counts = np.bincount(train_labels.astype(int), minlength=2)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = [class_weights[int(label)] for label in train_labels]

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Classes train [nofire, fire]:", class_counts.tolist())
print("Batches train:", len(train_loader), "| Batches validation:", len(val_loader))


In [ ]:
def show_samples(n=8):
    indices = np.random.choice(len(all_paths), size=min(n, len(all_paths)), replace=False)
    cols = min(4, len(indices))
    rows = int(np.ceil(len(indices) / cols))

    plt.figure(figsize=(4 * cols, 3 * rows))
    for i, idx in enumerate(indices):
        image = Image.open(all_paths[idx]).convert("RGB")
        label = "fire" if all_labels[idx] == 1 else "nofire"

        plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.title(label)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_samples()


## Construction du modèle CNN


In [ ]:
MODEL_NAME_CANDIDATES = [
    "convnextv2_tiny.fcmae_ft_in22k_in1k",
    "convnextv2_tiny.fcmae_ft_in1k",
    "convnextv2_tiny",
]


def create_convnextv2_tiny(dropout=0.20):
    last_error = None

    for model_name in MODEL_NAME_CANDIDATES:
        try:
            model = timm.create_model(
                model_name,
                pretrained=True,
                num_classes=1,
                drop_rate=dropout,
            )
            return model, model_name
        except Exception as error:
            last_error = error

    raise RuntimeError(f"Aucun modèle ConvNeXt V2 Tiny n'a pu être chargé : {last_error}")


model, MODEL_NAME_USED = create_convnextv2_tiny(dropout=DROPOUT)
model = model.to(DEVICE)


def count_parameters(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable


total_params, trainable_params = count_parameters(model)
print("Modèle utilisé:", MODEL_NAME_USED)
print("Paramètres totaux:", f"{total_params:,}")
print("Paramètres entraînables:", f"{trainable_params:,}")


In [ ]:
def set_requires_grad(module, value):
    for parameter in module.parameters():
        parameter.requires_grad = value


def get_head_module(model):
    if hasattr(model, "head"):
        return model.head
    return model.get_classifier()


def prepare_head_training(model):
    set_requires_grad(model, False)
    set_requires_grad(get_head_module(model), True)


def prepare_fine_tuning(model, unfreeze_stages=2):
    set_requires_grad(model, False)
    set_requires_grad(get_head_module(model), True)

    if hasattr(model, "stages"):
        for stage in list(model.stages)[-unfreeze_stages:]:
            set_requires_grad(stage, True)
    elif hasattr(model, "features"):
        for block in list(model.features)[-unfreeze_stages:]:
            set_requires_grad(block, True)
    else:
        for child in list(model.children())[-unfreeze_stages:]:
            set_requires_grad(child, True)


prepare_head_training(model)
total_params, trainable_params = count_parameters(model)
print("Paramètres entraînables après gel initial:", f"{trainable_params:,}", "/", f"{total_params:,}")


## Entraînement


In [ ]:
def compute_pos_weight(labels):
    labels = np.array(labels).astype(int)
    n_pos = np.sum(labels == 1)
    n_neg = np.sum(labels == 0)
    return torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)


def smooth_binary_targets(targets, eps):
    if eps <= 0:
        return targets
    return targets * (1.0 - eps) + 0.5 * eps


def get_autocast_context():
    return torch.amp.autocast(device_type="cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))


POS_WEIGHT = compute_pos_weight(train_labels).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
print("POS_WEIGHT:", round(POS_WEIGHT.item(), 4))


In [ ]:
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, label_smoothing=0.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    losses, probabilities, targets_all = [], [], []

    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True).view(-1, 1)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with get_autocast_context():
                logits = model(images)
                if logits.ndim == 1:
                    logits = logits.view(-1, 1)

                targets_for_loss = smooth_binary_targets(targets, label_smoothing)
                loss = criterion(logits, targets_for_loss)

            if is_train:
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

        losses.append(loss.item() * images.size(0))
        probabilities.extend(torch.sigmoid(logits.detach()).cpu().numpy().reshape(-1).tolist())
        targets_all.extend(targets.detach().cpu().numpy().reshape(-1).astype(int).tolist())

    avg_loss = np.sum(losses) / len(loader.dataset)
    probabilities = np.array(probabilities)
    targets_all = np.array(targets_all)
    predictions = (probabilities >= DEFAULT_THRESHOLD).astype(int)

    metrics = {
        "loss": avg_loss,
        "accuracy": accuracy_score(targets_all, predictions),
        "precision_fire": precision_score(targets_all, predictions, zero_division=0),
        "recall_fire": recall_score(targets_all, predictions, zero_division=0),
        "f1_fire": f1_score(targets_all, predictions, zero_division=0),
    }

    try:
        metrics["auc"] = roc_auc_score(targets_all, probabilities)
    except Exception:
        metrics["auc"] = np.nan

    return metrics, targets_all, probabilities


In [ ]:
def update_best_model(current_loss, epoch_index, model, best_state, best_val_loss, best_epoch):
    if current_loss < best_val_loss:
        best_state = {
            "model_state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
            "best_epoch": epoch_index,
            "best_val_loss": current_loss,
            "threshold": DEFAULT_THRESHOLD,
            "config": {
                "model": "convnextv2_tiny",
                "model_name_used": MODEL_NAME_USED,
                "image_size": IMAGE_SIZE,
                "batch_size": BATCH_SIZE,
                "lr_head": LR_HEAD,
                "lr_fine": LR_FINE,
                "weight_decay": WEIGHT_DECAY,
                "dropout": DROPOUT,
                "label_smoothing": LABEL_SMOOTHING,
                "unfreeze_stages": UNFREEZE_STAGES,
            },
        }
        return best_state, current_loss, epoch_index, True

    return best_state, best_val_loss, best_epoch, False


def train_phase(phase_name, epochs, optimizer, scheduler, start_epoch, best_state, best_val_loss, best_epoch, patience_counter):
    for _ in range(epochs):
        start_epoch += 1

        train_metrics, _, _ = run_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
            scaler=scaler,
            label_smoothing=LABEL_SMOOTHING,
        )

        val_metrics, y_val, probs_val = run_one_epoch(model, val_loader, criterion)
        scheduler.step(val_metrics["loss"])

        history.append({
            "epoch": start_epoch,
            "phase": phase_name,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_precision_fire": train_metrics["precision_fire"],
            "train_recall_fire": train_metrics["recall_fire"],
            "train_f1_fire": train_metrics["f1_fire"],
            "train_auc": train_metrics["auc"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision_fire": val_metrics["precision_fire"],
            "val_recall_fire": val_metrics["recall_fire"],
            "val_f1_fire": val_metrics["f1_fire"],
            "val_auc": val_metrics["auc"],
            "lr": optimizer.param_groups[0]["lr"],
        })

        print(
            f"Epoch {start_epoch:02d} [{phase_name}] "
            f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['accuracy']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
            f"val_f1={val_metrics['f1_fire']:.4f} recall_fire={val_metrics['recall_fire']:.4f}"
        )

        best_state, best_val_loss, best_epoch, improved = update_best_model(
            val_metrics["loss"], start_epoch, model, best_state, best_val_loss, best_epoch
        )
        patience_counter = 0 if improved else patience_counter + 1

        if phase_name == "fine" and patience_counter >= PATIENCE:
            print("Early stopping déclenché.")
            break

    return start_epoch, best_state, best_val_loss, best_epoch, patience_counter


In [ ]:
history = []
best_val_loss = float("inf")
best_state = None
best_epoch = -1
patience_counter = 0
epoch_index = 0

scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))
start_time = time.time()

prepare_head_training(model)
optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.4, patience=2)

print("Phase 1 : entraînement de la tête de classification")
epoch_index, best_state, best_val_loss, best_epoch, patience_counter = train_phase(
    "head", HEAD_EPOCHS, optimizer, scheduler, epoch_index, best_state, best_val_loss, best_epoch, patience_counter
)

prepare_fine_tuning(model, unfreeze_stages=UNFREEZE_STAGES)
optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR_FINE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.4, patience=2)

total_params, trainable_params = count_parameters(model)
print("\nPhase 2 : fine-tuning")
print("Paramètres entraînables:", f"{trainable_params:,}", "/", f"{total_params:,}")

epoch_index, best_state, best_val_loss, best_epoch, patience_counter = train_phase(
    "fine", FINE_EPOCHS, optimizer, scheduler, epoch_index, best_state, best_val_loss, best_epoch, patience_counter
)

elapsed_min = (time.time() - start_time) / 60
history_df = pd.DataFrame(history)
history_path = os.path.join(RESULTS_DIR, "convnextv2_tiny_history.csv")
history_df.to_csv(history_path, index=False)

print("\nTemps total d'entraînement:", round(elapsed_min, 2), "minutes")
print("Historique sauvegardé:", history_path)


## Évaluation


In [ ]:
def plot_history(history_df):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history_df["epoch"], history_df["train_loss"], label="Train loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("ConvNeXt V2 Tiny — Loss")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(history_df["epoch"], history_df["train_accuracy"], label="Train accuracy")
    plt.plot(history_df["epoch"], history_df["val_accuracy"], label="Validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("ConvNeXt V2 Tiny — Accuracy")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, "convnextv2_tiny_training_curves.png")
    plt.savefig(fig_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Courbes sauvegardées:", fig_path)

plot_history(history_df)


In [ ]:
if best_state is not None:
    model.load_state_dict(best_state["model_state_dict"])
else:
    raise RuntimeError("Aucun meilleur modèle n'a été enregistré pendant l'entraînement.")

model_path = os.path.join(MODELS_DIR, "convnextv2_tiny_best.pt")
torch.save(best_state, model_path)

print("Meilleur epoch:", best_epoch)
print("Meilleure validation loss:", round(best_val_loss, 4))
print("Modèle sauvegardé:", model_path)


In [ ]:
def optimize_threshold(y_true, probabilities):
    rows = []
    thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)

    for threshold_value in thresholds:
        predictions = (probabilities >= threshold_value).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()

        rows.append({
            "threshold": threshold_value,
            "accuracy": accuracy_score(y_true, predictions),
            "precision_fire": precision_score(y_true, predictions, zero_division=0),
            "recall_fire": recall_score(y_true, predictions, zero_division=0),
            "f1_fire": f1_score(y_true, predictions, zero_division=0),
            "false_negative_fire": fn,
            "false_positive_fire": fp,
        })

    threshold_df = pd.DataFrame(rows)
    threshold_df = threshold_df.sort_values(
        by=["f1_fire", "recall_fire", "accuracy", "false_negative_fire"],
        ascending=[False, False, False, True],
    )
    return threshold_df.iloc[0].to_dict(), threshold_df


In [ ]:
val_metrics, y_val, probs_val = run_one_epoch(model, val_loader, criterion)
best_threshold, threshold_df = optimize_threshold(y_val, probs_val)
threshold = float(best_threshold["threshold"])

threshold_path = os.path.join(RESULTS_DIR, "convnextv2_tiny_thresholds.csv")
threshold_df.to_csv(threshold_path, index=False)

preds_opt = (probs_val >= threshold).astype(int)
cm = confusion_matrix(y_val, preds_opt, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

final_results = {
    "model": "ConvNeXt V2 Tiny",
    "model_name_used": MODEL_NAME_USED,
    "best_epoch": best_epoch,
    "best_val_loss": best_val_loss,
    "threshold": threshold,
    "val_accuracy": accuracy_score(y_val, preds_opt),
    "val_precision_fire": precision_score(y_val, preds_opt, zero_division=0),
    "val_recall_fire": recall_score(y_val, preds_opt, zero_division=0),
    "val_f1_fire": f1_score(y_val, preds_opt, zero_division=0),
    "val_auc": roc_auc_score(y_val, probs_val),
    "false_negative_fire": int(fn),
    "false_positive_fire": int(fp),
    "true_fire": int(tp + fn),
    "true_nofire": int(tn + fp),
    "time_min": elapsed_min,
    "model_path": model_path,
}

final_df = pd.DataFrame([final_results])
final_path = os.path.join(RESULTS_DIR, "convnextv2_tiny_final_results.csv")
final_df.to_csv(final_path, index=False)

display(final_df)
print("\nClassification report:")
print(classification_report(y_val, preds_opt, target_names=["nofire", "fire"], zero_division=0))
print("Résultats sauvegardés:", final_path)


In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title(f"ConvNeXt V2 Tiny — Matrice de confusion\nThreshold = {threshold}")
plt.xticks([0, 1], ["nofire", "fire"])
plt.yticks([0, 1], ["nofire", "fire"])
plt.xlabel("Prédiction")
plt.ylabel("Vraie classe")

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)

plt.colorbar()
plt.tight_layout()

cm_path = os.path.join(FIGURES_DIR, "convnextv2_tiny_confusion_matrix.png")
plt.savefig(cm_path, dpi=160, bbox_inches="tight")
plt.show()
print("Matrice sauvegardée:", cm_path)


In [ ]:
last_epoch = history_df.iloc[-1]
train_acc = last_epoch["train_accuracy"]
val_acc = last_epoch["val_accuracy"]
gap = train_acc - val_acc

if history_df["train_accuracy"].max() < 0.85 and history_df["val_accuracy"].max() < 0.85:
    diagnosis = "Underfitting probable : performances faibles sur train et validation."
elif gap > 0.08 and train_acc > 0.93:
    diagnosis = "Overfitting probable : écart important entre train et validation."
elif history_df["val_loss"].iloc[-1] > history_df["val_loss"].min() * 1.20 and gap > 0.05:
    diagnosis = "Overfitting léger possible : la validation loss augmente."
else:
    diagnosis = "Bon ajustement : les performances train et validation sont proches."

print("Train accuracy finale:", round(train_acc, 4))
print("Validation accuracy finale:", round(val_acc, 4))
print("Écart train-validation:", round(gap, 4))
print("Diagnostic:", diagnosis)


## Résultats / prédictions


In [ ]:
def denormalize(image_tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    image = image_tensor.cpu() * std + mean
    image = torch.clamp(image, 0, 1)
    return image.permute(1, 2, 0).numpy()


wrong_indices = np.where(preds_opt != y_val)[0]
print("Images mal classées:", len(wrong_indices))

if len(wrong_indices) > 0:
    n = min(len(wrong_indices), 12)
    plt.figure(figsize=(14, 4 * ((n + 3) // 4)))

    for k, idx in enumerate(wrong_indices[:n]):
        image_tensor, label = val_ds[idx]
        probability = probs_val[idx]
        prediction = preds_opt[idx]

        plt.subplot((n + 3) // 4, 4, k + 1)
        plt.imshow(denormalize(image_tensor))
        plt.axis("off")
        plt.title(
            f"Vrai: {'fire' if label.item() == 1 else 'nofire'}\n"
            f"Prédit: {'fire' if prediction == 1 else 'nofire'} | p_fire={probability:.3f}"
        )

    plt.tight_layout()
    wrong_path = os.path.join(FIGURES_DIR, "convnextv2_tiny_wrong_predictions.png")
    plt.savefig(wrong_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Figure sauvegardée:", wrong_path)
else:
    print("Aucune erreur sur la validation avec le threshold optimisé.")


In [ ]:
tta_transform_original = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

tta_transform_flip = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_ds_tta_original = ForestFireDataset(val_paths, val_labels, transform=tta_transform_original)
val_ds_tta_flip = ForestFireDataset(val_paths, val_labels, transform=tta_transform_flip)

val_loader_tta_original = DataLoader(val_ds_tta_original, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
val_loader_tta_flip = DataLoader(val_ds_tta_flip, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

_, y_tta, probs_original = run_one_epoch(model, val_loader_tta_original, criterion)
_, _, probs_flip = run_one_epoch(model, val_loader_tta_flip, criterion)

probs_tta = (probs_original + probs_flip) / 2.0
best_tta_threshold, tta_threshold_df = optimize_threshold(y_tta, probs_tta)
tta_threshold = float(best_tta_threshold["threshold"])
preds_tta = (probs_tta >= tta_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_tta, preds_tta, labels=[0, 1]).ravel()
tta_results = {
    "model": "ConvNeXt V2 Tiny + TTA",
    "threshold": tta_threshold,
    "accuracy": accuracy_score(y_tta, preds_tta),
    "precision_fire": precision_score(y_tta, preds_tta, zero_division=0),
    "recall_fire": recall_score(y_tta, preds_tta, zero_division=0),
    "f1_fire": f1_score(y_tta, preds_tta, zero_division=0),
    "auc": roc_auc_score(y_tta, probs_tta),
    "false_negative_fire": int(fn),
    "false_positive_fire": int(fp),
}

tta_path = os.path.join(RESULTS_DIR, "convnextv2_tiny_tta_results.csv")
pd.DataFrame([tta_results]).to_csv(tta_path, index=False)

display(pd.DataFrame([tta_results]))
print("Résultats TTA sauvegardés:", tta_path)


In [ ]:
def collect_test_images(testing_dir):
    paths = []

    if not os.path.exists(testing_dir):
        print("Dossier Testing introuvable:", testing_dir)
        return np.array(paths)

    for root, _, files in os.walk(testing_dir):
        for file_name in files:
            image_path = os.path.join(root, file_name)
            if is_image_file(image_path):
                paths.append(image_path)

    return np.array(paths)


test_paths = collect_test_images(TESTING_DIR)
print("Images Testing:", len(test_paths))

if len(test_paths) > 0:
    test_ds = ForestFireDataset(test_paths, labels=None, transform=val_transform)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model.eval()
    rows = []

    with torch.no_grad():
        for images, paths in test_loader:
            images = images.to(DEVICE, non_blocking=True)

            with get_autocast_context():
                logits = model(images)
                if logits.ndim == 1:
                    logits = logits.view(-1, 1)
                probabilities = torch.sigmoid(logits).cpu().numpy().reshape(-1)

            for image_path, probability in zip(paths, probabilities):
                prediction = int(probability >= threshold)
                rows.append({
                    "filename": os.path.basename(image_path),
                    "path": image_path,
                    "prob_fire": float(probability),
                    "prediction": "fire" if prediction == 1 else "nofire",
                    "threshold": threshold,
                    "model": "ConvNeXt V2 Tiny",
                    "model_name_used": MODEL_NAME_USED,
                })

    pred_df = pd.DataFrame(rows)
    pred_path = os.path.join(PREDICTIONS_DIR, "testing_predictions_convnextv2_tiny.csv")
    pred_df.to_csv(pred_path, index=False)

    print("Prédictions sauvegardées:", pred_path)
    display(pred_df.head(20))
else:
    print("Aucune image Testing trouvée.")


## Conclusion

Le modèle **ConvNeXt V2 Tiny pré-entraîné** a été adapté pour une classification binaire des images en **fire** et **nofire**. L’entraînement est réalisé en deux phases : entraînement de la tête de classification, puis fine-tuning des derniers blocs du réseau.

L’évaluation repose sur l’accuracy, la précision, le recall, le F1-score, l’AUC et la matrice de confusion. Dans ce contexte, le **recall de la classe fire** est particulièrement important, car l’objectif prioritaire est de limiter les faux négatifs, c’est-à-dire les images d’incendie non détectées.
